In [1]:
import numpy as np
from pynq import Overlay, allocate
import time

class SVCAccelerator:
    """
    Hardware-accelerated SVC classifier for PYNQ
    """
    def __init__(self, bitstream_path, model_params=None, classes_file=None):
        """
        Initialize the SVC accelerator
        
        Args:
            bitstream_path: Path to .bit or .xsa file
            model_params: Path to .npz file with model parameters OR dict
            classes_file: Path to text file with class labels (one per line)
        """
        # Load overlay
        print("Loading bitstream...")
        self.overlay = Overlay(bitstream_path)
        self.accelerator = self.overlay.svc_predict_0
        
        # Model configuration
        self.n_features = None
        self.n_classes = None
        self.classes = None
        
        # Allocate DDR buffers (maximum sizes)
        self.MAX_FEATURES = 256
        self.MAX_CLASSES = 32
        
        print("Allocating DDR buffers...")
        self.x_buf = allocate(shape=(self.MAX_FEATURES,), dtype=np.float32)
        self.coef_buf = allocate(shape=(self.MAX_CLASSES * self.MAX_FEATURES,), dtype=np.float32)
        self.intercept_buf = allocate(shape=(self.MAX_CLASSES,), dtype=np.float32)
        self.mean_buf = allocate(shape=(self.MAX_FEATURES,), dtype=np.float32)
        self.std_buf = allocate(shape=(self.MAX_FEATURES,), dtype=np.float32)
        self.scores_buf = allocate(shape=(self.MAX_CLASSES,), dtype=np.float32)
        
        # Load model if provided
        if model_params is not None:
            if isinstance(model_params, str):
                # Load from .npz file
                self.load_model(model_params, classes_file)
            else:
                # Load from dict
                self.load_model_from_dict(model_params)
        
        print("✓ SVC Accelerator initialized")
    
    def load_model(self, npz_path, classes_file=None):
        """Load model parameters from .npz file"""
        print(f"Loading model from {npz_path}...")
        data = np.load(npz_path, allow_pickle=False)
        
        # Load class labels from text file
        if classes_file is None:
            # Try to find it automatically
            classes_file = npz_path.replace('.npz', '_classes.txt')
            if not os.path.exists(classes_file):
                classes_file = 'svc_classes.txt'
        
        print(f"Loading classes from {classes_file}...")
        classes = []
        with open(classes_file, 'r') as f:
            for line in f:
                label = line.strip()
                if label:  # Skip empty lines
                    classes.append(label)
        
        self.load_model_from_dict({
            'coef': data['coef'],
            'intercept': data['intercept'],
            'mean': data['mean'],
            'std': data['std'],
            'classes': np.array(classes)
        })
    
    def load_model_from_dict(self, params):
        """Load model parameters from dictionary"""
        coef = params['coef'].astype(np.float32)
        intercept = params['intercept'].astype(np.float32)
        mean = params['mean'].astype(np.float32)
        std = params['std'].astype(np.float32)
        
        self.n_classes = coef.shape[0]
        self.n_features = coef.shape[1]
        self.classes = params['classes']
        
        print(f"  Features: {self.n_features}")
        print(f"  Classes: {self.n_classes}")
        print(f"  Class labels: {list(self.classes)}")
        
        # Validate sizes
        if self.n_features > self.MAX_FEATURES:
            raise ValueError(f"n_features {self.n_features} exceeds MAX_FEATURES {self.MAX_FEATURES}")
        if self.n_classes > self.MAX_CLASSES:
            raise ValueError(f"n_classes {self.n_classes} exceeds MAX_CLASSES {self.MAX_CLASSES}")
        
        # Copy to DDR buffers
        self.coef_buf[:self.n_classes * self.n_features] = coef.flatten()
        self.intercept_buf[:self.n_classes] = intercept
        self.mean_buf[:self.n_features] = mean
        self.std_buf[:self.n_features] = std
        
        # Sync to device
        self.coef_buf.sync_to_device()
        self.intercept_buf.sync_to_device()
        self.mean_buf.sync_to_device()
        self.std_buf.sync_to_device()
        
        print("✓ Model loaded to FPGA")
    
    def predict(self, x, return_confidence=False):
        """
        Predict class for a single sample
        
        Args:
            x: Input features (numpy array of shape (n_features,))
            return_confidence: If True, return confidence scores
            
        Returns:
            If return_confidence=False: predicted class label
            If return_confidence=True: dict with 'class', 'confidence', 'margin', 'all_scores'
        """
        if self.n_features is None:
            raise RuntimeError("Model not loaded. Call load_model() first.")
        
        x = np.array(x, dtype=np.float32)
        if x.shape[0] != self.n_features:
            raise ValueError(f"Expected {self.n_features} features, got {x.shape[0]}")
        
        # Copy input to buffer
        self.x_buf[:self.n_features] = x
        self.x_buf.sync_to_device()
        
        # Configure accelerator
        self.accelerator.write(0x10, self.x_buf.physical_address & 0xFFFFFFFF)  # x_in_1
        self.accelerator.write(0x14, (self.x_buf.physical_address >> 32) & 0xFFFFFFFF)  # x_in_2
        
        self.accelerator.write(0x1c, self.coef_buf.physical_address & 0xFFFFFFFF)  # coef_1
        self.accelerator.write(0x20, (self.coef_buf.physical_address >> 32) & 0xFFFFFFFF)  # coef_2
        
        self.accelerator.write(0x28, self.intercept_buf.physical_address & 0xFFFFFFFF)  # intercept_1
        self.accelerator.write(0x2c, (self.intercept_buf.physical_address >> 32) & 0xFFFFFFFF)  # intercept_2
        
        self.accelerator.write(0x34, self.mean_buf.physical_address & 0xFFFFFFFF)  # mean_1
        self.accelerator.write(0x38, (self.mean_buf.physical_address >> 32) & 0xFFFFFFFF)  # mean_2
        
        self.accelerator.write(0x40, self.std_buf.physical_address & 0xFFFFFFFF)  # std_1
        self.accelerator.write(0x44, (self.std_buf.physical_address >> 32) & 0xFFFFFFFF)  # std_2
        
        self.accelerator.write(0x6c, self.scores_buf.physical_address & 0xFFFFFFFF)  # scores_out_1
        self.accelerator.write(0x70, (self.scores_buf.physical_address >> 32) & 0xFFFFFFFF)  # scores_out_2
        
        self.accelerator.write(0x4c, self.n_features)  # n_features
        self.accelerator.write(0x54, self.n_classes)   # n_classes
        
        # Start computation (write to CTRL register, bit 0 = AP_START)
        self.accelerator.write(0x00, 0x01)
        
        # Wait for completion (poll bit 1 = AP_DONE)
        while (self.accelerator.read(0x00) & 0x02) == 0:
            pass
        
        # Read results
        class_idx = self.accelerator.read(0x5c)  # class_out
        max_conf = np.frombuffer(self.accelerator.read(0x78).to_bytes(4, 'little'), dtype=np.float32)[0]
        second_conf = np.frombuffer(self.accelerator.read(0x88).to_bytes(4, 'little'), dtype=np.float32)[0]
        
        # Get all scores
        self.scores_buf.sync_from_device()
        all_scores = self.scores_buf[:self.n_classes].copy()
        
        # Map to class label
        predicted_class = self.classes[class_idx]
        
        if not return_confidence:
            return predicted_class
        else:
            margin = max_conf - second_conf
            return {
                'class': predicted_class,
                'class_idx': class_idx,
                'confidence': float(max_conf),
                'margin': float(margin),
                'all_scores': all_scores,
                'uncertainty': float(1.0 / (1.0 + margin))
            }
    
    def predict_batch(self, X, return_confidence=False):
        """
        Predict classes for multiple samples
        
        Args:
            X: Input features (numpy array of shape (n_samples, n_features))
            return_confidence: If True, return confidence info for each sample
            
        Returns:
            If return_confidence=False: array of predicted class labels
            If return_confidence=True: list of dicts with prediction info
        """
        results = []
        for i in range(X.shape[0]):
            result = self.predict(X[i], return_confidence=return_confidence)
            results.append(result)
        
        if not return_confidence:
            return np.array(results)
        else:
            return results
    
    def benchmark(self, x, num_iterations=1000):
        """
        Benchmark inference speed
        
        Args:
            x: Sample input
            num_iterations: Number of inferences to run
            
        Returns:
            Dict with timing statistics
        """
        print(f"Running {num_iterations} inferences...")
        
        times = []
        for i in range(num_iterations):
            start = time.time()
            _ = self.predict(x)
            end = time.time()
            times.append((end - start) * 1000)  # Convert to ms
        
        times = np.array(times)
        
        return {
            'mean_ms': np.mean(times),
            'std_ms': np.std(times),
            'min_ms': np.min(times),
            'max_ms': np.max(times),
            'throughput_fps': 1000.0 / np.mean(times)
        }
    
    def __del__(self):
        """Cleanup buffers"""
        if hasattr(self, 'x_buf'):
            del self.x_buf
            del self.coef_buf
            del self.intercept_buf
            del self.mean_buf
            del self.std_buf
            del self.scores_buf

In [ ]:
# ============================================
# TEST EXAMPLE WITH RANDOM DATA

# design_2:
# self.MAX_FEATURES = 256
# self.MAX_CLASSES = 32
# ============================================
import os

if __name__ == "__main__":
    # 2. On PYNQ board
    # Initialize accelerator
    svc_hw = SVCAccelerator(
        bitstream_path='svc_design_2.xsa',
        model_params='svc_model_mqtt_median_906_10features.npz'
    )
    
    # Single prediction
    sample = np.random.randn(10)  # single sample: (X num_features) (depending on svc_model.npz)
    predicted_class = svc_hw.predict(sample)
    print(f"Predicted: {predicted_class}")
    
    # With confidence
    result = svc_hw.predict(sample, return_confidence=True)
    print(f"Class: {result['class']}")
    print(f"Confidence: {result['confidence']:.3f}")
    print(f"Margin: {result['margin']:.3f}")
    print(f"Top 3 classes: {np.argsort(result['all_scores'])[-3:][::-1]}")
    
    # Batch prediction 
    X_batch = np.random.randn(10, 10) # batch samples: (num_samples, X num_features) (depending on svc_model.npz)
    predictions = svc_hw.predict_batch(X_batch)
    print(f"Batch predictions: {predictions}")
    
    # Benchmark
    stats = svc_hw.benchmark(sample, num_iterations=1000)
    print(f"\nBenchmark Results:")
    print(f"  Mean latency: {stats['mean_ms']:.3f} ms")
    print(f"  Throughput: {stats['throughput_fps']:.1f} FPS")

Loading bitstream...


Allocating DDR buffers...
Loading model from svc_model_mqtt_median_826.npz...
Loading classes from svc_classes.txt...
  Features: 10
  Classes: 11
  Class labels: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'REST']
✓ Model loaded to FPGA
✓ SVC Accelerator initialized
Running 1000 inferences...

Benchmark Results:
  Mean latency: 0.528 ms
  Throughput: 1893.7 FPS


In [3]:
# ============================================
# EXAMPLE WITH CSV (RECORDED) DATA
# ============================================

import pandas as pd

# ====================================================================
# LOAD CSV RECORDED DATA
# ====================================================================

df = pd.read_csv("data/glove_data_mqtt_11.csv")
df.columns = df.columns.str.strip()

# 1. Set y to be LABEL
y = df["LABEL"].to_numpy()                # shape: (num_samples,)
df_features = df.drop(columns=["LABEL"])  # remove label column

# 2. Set X to be (num_samples, num_timesteps, num_features) 
num_timesteps = 100

feature_names = ["roll", "pitch", "yaw", "ax", "ay", "az", "t0", "t1", "f0", "f1", "f2", "f3"]
ignored_features = ["yaw", "t1"]  # feature to ignore
feature_names = [f for f in feature_names if f not in ignored_features]
num_features = len(feature_names)

columns_ordered = []
for i in range(1, num_timesteps + 1):
    for f in feature_names:
        if (f.startswith("t1")):
            columns_ordered.append(f"{f}{i}")
        else:
            # naming convention: f0_1, f1_1, etc., but t1{1} (no underscore in csv file) <= what a mistake :(
            columns_ordered.append(f"{f}_{i}")

df_features = df_features[columns_ordered]

data = df_features.to_numpy(dtype=np.float16)  # (num_samples, 220)
X = data.reshape((-1, num_timesteps, num_features))  # (num_samples, 20, 12?)
num_samples = X.shape[0]

# ====================================================================
# FEATURE EXTRACTION
# ====================================================================
def extract_features(X):
    """
    X: shape:
    (num_samples, timesteps, num_features) for batch samples 
    or (timesteps, num_features) for single sample
    
    returns: feature matrix (num_samples, num_engineered_features)
    """
    # Ensure X has batch dimension
    if X.ndim == 2:
        X = X[np.newaxis, ...]  # (1, timesteps, features)
    
    num_samples, T, F = X.shape
    all_features = []

    for i in range(num_samples):
        x = X[i].astype(np.float32)
        x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
        feats = []

        # Median and std
        feats.extend(np.median(x, axis=0))  # F features
        feats.extend(np.std(x, axis=0))     # F features

        # Roll-pitch correlation
        roll, pitch = x[:, 0], x[:, 1]
        feats.append(np.corrcoef(roll, pitch)[0,1] if np.std(roll) > 1e-8 and np.std(pitch) > 1e-8 else 0)

        # Acceleration magnitude features
        ax, ay, az = x[:, 2], x[:, 3], x[:, 4]
        acc_mag = np.sqrt(ax**2 + ay**2 + az**2)
        feats.extend([np.mean(acc_mag), np.std(acc_mag), np.max(acc_mag), np.median(acc_mag)])

        # Flex features (assuming last 5 columns)
        f = x[:, -5:]  # last 5 columns are flex
        feats.extend(np.std(f, axis=0))
        feats.extend(np.median(f, axis=0))
        feats.append(np.mean(np.median(f, axis=1)))

        # Flex states (OP=0, PR=0.5, CR=1)
        flex_median = np.median(f, axis=0)
        flex_state = np.zeros_like(flex_median)
        flex_state[(flex_median >= 35) & (flex_median < 90)] = 0.5
        flex_state[flex_median >= 90] = 1
        feats.extend(flex_state)

        # Pairwise finger correlations
        if np.all(np.std(f, axis=0) > 1e-8):
            corr = np.corrcoef(f.T)[np.triu_indices(5, k=1)]
            feats.extend(corr)
        else:
            feats.extend([0]*10)

        # Pairwise differences (mean + std)
        for i in range(f.shape[1]):
            for j in range(i+1, f.shape[1]):
                diff = f[:, i] - f[:, j]
                feats.append(np.mean(diff))
                feats.append(np.std(diff))

        all_features.append(feats)

    X_new = np.array(all_features, dtype=np.float32)
    print(f"Extracted features shape (num_samples, num_features): {X_new.shape}")
    return X_new

# X = extract_features(X)
X = np.median(X, axis=1)

print(f"X shape: {X.shape}")

X shape: (45, 10)


In [4]:
# ====================================================================
# INFERENCE EVERY SINGLE SAMPLE FROM CSV
# ====================================================================

num_samples = X.shape[0]
num_correct = 0

for i in range(0, num_samples):
    sample = X[i]
    result = svc_hw.predict(sample, return_confidence=True)
    
    print(f"Sample {i}:")
    print(f"True Class: {y[i]}")
    print(f"Predicted Class: {result['class']}")
    
    if y[i] == result['class']:
        num_correct += 1
        
    print(f"Confidence: {result['confidence']:.3f}")
    print(f"Margin: {result['margin']:.3f}")
    
    top3_idx = np.argsort(result['all_scores'])[-3:][::-1]  # top 3 indices
    top3_labels = [svc_hw.classes[i] for i in top3_idx]     # map indices to letters
    print(f"Top 3 classes: {top3_labels}\n")

print(num_correct/num_samples)

Sample 0:
True Class: A
Predicted Class: A
Confidence: 0.480
Margin: 1.038
Top 3 classes: ['A', 'E', 'I']

Sample 1:
True Class: B
Predicted Class: B
Confidence: -0.015
Margin: 0.397
Top 3 classes: ['B', 'I', 'E']

Sample 2:
True Class: C
Predicted Class: C
Confidence: -0.110
Margin: 0.320
Top 3 classes: ['C', 'E', 'I']

Sample 3:
True Class: D
Predicted Class: D
Confidence: 0.419
Margin: 1.116
Top 3 classes: ['D', 'C', 'F']

Sample 4:
True Class: E
Predicted Class: E
Confidence: 0.925
Margin: 1.562
Top 3 classes: ['E', 'F', 'REST']

Sample 5:
True Class: F
Predicted Class: F
Confidence: 0.490
Margin: 1.291
Top 3 classes: ['F', 'A', 'I']

Sample 6:
True Class: G
Predicted Class: G
Confidence: 0.627
Margin: 1.374
Top 3 classes: ['G', 'H', 'REST']

Sample 7:
True Class: H
Predicted Class: G
Confidence: 0.066
Margin: 0.418
Top 3 classes: ['G', 'H', 'REST']

Sample 8:
True Class: I
Predicted Class: I
Confidence: 0.475
Margin: 1.189
Top 3 classes: ['I', 'C', 'J']

Sample 9:
True Class: J
Pr